# Experiment 1 — Paper Model: Leakage Demo + Honest Baseline

**Notebook replaces** `model.py` + `train.py`.

| | Split | Expected Acc | Meaning |
|---|---|---|---|
| **1A** | Slice-level (random) | ~99% 🚨 | Artificially inflated — demonstrates leakage |
| **1B** | Sequential per-class | ~82–87% ✅ | Honest result used in thesis |

**Pipeline**
1. Configuration & imports
2. Model definition (`ConcatCNNAttention`)
3. Dataset & DataLoaders
4. Training helpers
5. Experiment 1A — Slice-level split *(leakage demo)*
6. Experiment 1B — Sequential split *(honest baseline)*
7. Leakage comparison table


---
## 0 · Configuration & Imports

In [ ]:
import os, sys, json, random, csv, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score
)
from PIL import Image

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = "/home/metal/Experiment/data/The IQ-OTHNCCD lung cancer dataset"
SPLITS_CSV   = Path("../../00_dataset/splits.csv")   # generated by preprocessing.ipynb
RESULTS_DIR  = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Dataset ──────────────────────────────────────────────────────────────────
CLASS_FOLDERS  = ["Benign cases", "Malignant cases", "Normal cases"]
CLASS_TO_LABEL = {"Benign cases": 0, "Malignant cases": 1, "Normal cases": 2}
LABEL_TO_CLASS = {0: "Benign",    1: "Malignant",      2: "Normal"}
NUM_CLASSES    = 3

# ── Training hyper-parameters ────────────────────────────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 32
LR           = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
WARMUP_EPOCHS= 5
EARLY_STOP_PAT = 10
NUM_WORKERS  = 4
IMG_SIZE     = 224
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"Dataset: {DATASET_ROOT}")
print(f"Splits : {SPLITS_CSV.resolve()}")


---
## 1 · Model Definition — Concatenated CNN-Attention

Reproduces the architecture from *"An End-to-End Concatenated CNN Attention Model
for the Classification of Lung Cancer"* (IQ-OTH/NCCD dataset).

```
Input (1×224×224)
    ├─ Branch 3×3 ──► (256×14×14)
    └─ Branch 5×5 ──► (256×14×14)
            │ concat
        (512×14×14)
            │ Channel Attention (SE)
        GlobalAvgPool → (512,)
            │ Dropout → FC(256) → FC(3)
        Logits
```


In [ ]:
class ConvBnRelu(nn.Module):
    """Conv → BatchNorm → ReLU."""
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class CNNBranch(nn.Module):
    """Single-kernel CNN feature extractor (4 stages, each halving spatial size)."""
    def __init__(self, in_channels=1, kernel=3):
        super().__init__()
        p = kernel // 2
        self.features = nn.Sequential(
            ConvBnRelu(in_channels,  32, kernel=kernel, padding=p),
            ConvBnRelu(32,           32, kernel=kernel, padding=p),
            nn.MaxPool2d(2, 2),          # 224 → 112
            ConvBnRelu(32, 64, kernel=kernel, padding=p),
            ConvBnRelu(64, 64, kernel=kernel, padding=p),
            nn.MaxPool2d(2, 2),          # 112 → 56
            ConvBnRelu(64,  128, kernel=kernel, padding=p),
            ConvBnRelu(128, 128, kernel=kernel, padding=p),
            nn.MaxPool2d(2, 2),          # 56 → 28
            ConvBnRelu(128, 256, kernel=kernel, padding=p),
            ConvBnRelu(256, 256, kernel=kernel, padding=p),
            nn.MaxPool2d(2, 2),          # 28 → 14
        )
    def forward(self, x): return self.features(x)  # (B, 256, 14, 14)


class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, mid), nn.ReLU(inplace=True),
            nn.Linear(mid, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        B, C, _, _ = x.shape
        w = self.fc(self.gap(x).view(B, C)).view(B, C, 1, 1)
        return x * w


class ConcatCNNAttention(nn.Module):
    """
    End-to-End Concatenated CNN-Attention.
    Parallel 3×3 and 5×5 branches capture different receptive fields;
    channel attention re-weights the concatenated feature map.
    """
    def __init__(self, in_channels=1, num_classes=3, dropout=0.5):
        super().__init__()
        self.branch_3x3 = CNNBranch(in_channels, kernel=3)
        self.branch_5x5 = CNNBranch(in_channels, kernel=5)
        C = 512  # 256+256 after concat
        self.attention  = ChannelAttention(C)
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(C, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        x = torch.cat([self.branch_3x3(x), self.branch_5x5(x)], dim=1)
        x = self.pool(self.attention(x)).flatten(1)
        return self.classifier(x)


# Sanity check
_m = ConcatCNNAttention()
_n = sum(p.numel() for p in _m.parameters())
_o = _m(torch.randn(2, 1, 224, 224))
print(f"✅ ConcatCNNAttention | params={_n/1e6:.2f}M | output shape={_o.shape}")
del _m, _n, _o


---
## 2 · Dataset & DataLoader Helpers

In [ ]:
import re

# ── Record builder ───────────────────────────────────────────────────────────
def extract_case_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else abs(hash(filename)) % 100_000


def build_case_registry(root=DATASET_ROOT):
    root = Path(root)
    records = []
    for cls_folder in CLASS_FOLDERS:
        d = root / cls_folder
        if not d.exists(): print(f"⚠️  Not found: {d}"); continue
        lbl = CLASS_TO_LABEL[cls_folder]
        for f in sorted(d.iterdir()):
            if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                records.append({'image_path': str(f), 'class_name': cls_folder,
                                'label': lbl, 'case_id': extract_case_id(f.name),
                                'filename': f.name})
    return records


# ── Split strategies ─────────────────────────────────────────────────────────
def sequential_split(records):
    """Honest: sequential 70/15/15 per class (acquisition order, no leakage)."""
    by_cls = defaultdict(list)
    for r in records: by_cls[r['class_name']].append(r)
    out = []
    for cls, recs in by_cls.items():
        recs = sorted(recs, key=lambda r: r['case_id'])
        n = len(recs); n_tr = int(n*TRAIN_RATIO); n_v = int(n*VAL_RATIO)
        for i, r in enumerate(recs):
            r2 = dict(r)
            r2['split'] = 'train' if i < n_tr else ('val' if i < n_tr+n_v else 'test')
            out.append(r2)
    return out


def slice_level_split(records, seed=42):
    """Leaky: random 70/15/15 ignoring patient boundaries."""
    rng = random.Random(seed)
    shuffled = list(records); rng.shuffle(shuffled)
    n = len(shuffled); n_tr = int(n*TRAIN_RATIO); n_v = int(n*VAL_RATIO)
    out = []
    for i, r in enumerate(shuffled):
        r2 = dict(r)
        r2['split'] = 'train' if i < n_tr else ('val' if i < n_tr+n_v else 'test')
        out.append(r2)
    return out


def load_splits_csv(path=SPLITS_CSV):
    with open(path) as f:
        rows = list(csv.DictReader(f))
    for r in rows: r['label'] = int(r['label']); r['case_id'] = int(r['case_id'])
    return rows


def get_class_weights(records):
    train = [r for r in records if r['split'] == 'train']
    cnt   = Counter(r['label'] for r in train)
    total = sum(cnt.values())
    w = torch.zeros(NUM_CLASSES)
    for lbl, c in cnt.items(): w[lbl] = total / (NUM_CLASSES * c)
    return w


# ── Transforms ───────────────────────────────────────────────────────────────
def get_transforms(split, augment=False):
    base = [T.Resize((IMG_SIZE, IMG_SIZE)), T.Grayscale(1)]
    aug  = ([T.RandomHorizontalFlip(0.5), T.RandomRotation(15),
              T.RandomAffine(0, translate=(0.05,0.05)),
              T.ColorJitter(0.15, 0.15)]
            if split == 'train' and augment else [])
    return T.Compose(base + aug + [T.ToTensor(), T.Normalize([0.5],[0.5])])


# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class CTDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self):  return len(self.records)
    def __getitem__(self, i):
        r   = self.records[i]
        img = Image.open(r['image_path']).convert('RGB')
        return self.transform(img), r['label']


def make_loaders(records, batch_size=BATCH_SIZE, augment_train=False):
    loaders = {}
    for split in ['train', 'val', 'test']:
        sub  = [r for r in records if r['split'] == split]
        tf   = get_transforms(split, augment=(split=='train' and augment_train))
        ds   = CTDataset(sub, tf)
        loaders[split] = DataLoader(
            ds, batch_size=batch_size,
            shuffle=(split=='train'),
            num_workers=NUM_WORKERS, pin_memory=True
        )
    return loaders['train'], loaders['val'], loaders['test']


print("✅ Dataset helpers defined.")


---
## 3 · Training Engine

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


def warmup_schedule(optimizer, epoch, warmup_epochs, base_lr):
    """Linear LR warmup for the first `warmup_epochs` epochs."""
    if epoch < warmup_epochs:
        scale = (epoch + 1) / warmup_epochs
        for pg in optimizer.param_groups:
            pg['lr'] = base_lr * scale


def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip):
    model.train()
    loss_sum = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        if grad_clip: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum = correct = total = 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        probs  = F.softmax(logits, dim=1)
        preds  = logits.argmax(1)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (preds == labels).sum().item()
        total    += imgs.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
    return loss_sum/total, correct/total, all_preds, all_labels, all_probs


def compute_metrics(preds, labels, probs):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    report = classification_report(labels, preds, target_names=names, output_dict=True, zero_division=0)
    try:
        auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')
    metrics = {
        'accuracy' : report['accuracy'],
        'f1_macro' : report['macro avg']['f1-score'],
        'auc_macro': auc,
        'per_class': {n: report[n] for n in names},
    }
    return metrics


def plot_curves(train_losses, val_losses, train_accs, val_accs, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
    axes[0].set_title('Loss'); axes[0].legend()
    axes[1].plot(train_accs, label='Train'); axes[1].plot(val_accs, label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend()
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


def plot_confusion(preds, labels, save_path, title='Confusion Matrix'):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(names)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i,j], ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title); plt.colorbar(im); plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight'); plt.show()


def run_experiment(model, train_loader, val_loader, test_loader,
                   save_dir, model_name, class_weights,
                   epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                   grad_clip=GRAD_CLIP, warmup_epochs=WARMUP_EPOCHS,
                   early_stop_pat=EARLY_STOP_PAT, device=DEVICE,
                   smoke_test=False):
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    if smoke_test: epochs = 2

    model = model.to(device)
    crit  = nn.CrossEntropyLoss(weight=class_weights.to(device))
    opt   = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr*0.01)

    best_f1, patience_cnt = 0.0, 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    for epoch in range(epochs):
        warmup_schedule(opt, epoch, warmup_epochs, lr)
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, crit, device, grad_clip)
        vl_loss, vl_acc, vl_preds, vl_labels, vl_probs = evaluate(model, val_loader, crit, device)
        val_m = compute_metrics(vl_preds, vl_labels, vl_probs)
        sched.step()

        train_losses.append(tr_loss); val_losses.append(vl_loss)
        train_accs.append(tr_acc);   val_accs.append(vl_acc)

        print(f"[{epoch+1:3d}/{epochs}] "
              f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | "
              f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} f1={val_m['f1_macro']:.4f}")

        if val_m['f1_macro'] > best_f1:
            best_f1 = val_m['f1_macro']
            torch.save(model.state_dict(), save_dir / f"{model_name}_best.pth")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= early_stop_pat:
                print(f"  Early stopping at epoch {epoch+1}.")
                break

    # ── Final test evaluation ─────────────────────────────────────────────────
    model.load_state_dict(torch.load(save_dir / f"{model_name}_best.pth", map_location=device))
    _, _, te_preds, te_labels, te_probs = evaluate(model, test_loader, crit, device)
    test_m = compute_metrics(te_preds, te_labels, te_probs)

    plot_curves(train_losses, val_losses, train_accs, val_accs, save_dir / f"{model_name}_curves.png")
    plot_confusion(te_preds, te_labels, save_dir / f"{model_name}_confusion.png", title=model_name)

    with open(save_dir / "metrics.json", 'w') as f:
        json.dump(test_m, f, indent=2)

    return test_m


print("✅ Training engine defined.")


---
## 4 · Experiment 1A — Slice-Level Split *(Leakage Demo)* 🚨

> **Do NOT use this for thesis results.** This reproduces the inflated ~99% accuracy
> from the original paper, caused by randomly scattering slices from the same patient
> across train and test sets.


In [ ]:
print("=" * 68)
print("🚨 EXPERIMENT 1A — Slice-Level Random Split (Leakage Demo)")
print("   Expected: ~95–99% accuracy — this is the FAKE result")
print("=" * 68)

records_all   = build_case_registry()
records_slice = slice_level_split(records_all)

print("\nSplit distribution (slices):")
for split in ['train', 'val', 'test']:
    sub = [r for r in records_slice if r['split'] == split]
    print(f"  {split:<6}: {len(sub)} imgs | {dict(Counter(r['class_name'] for r in sub))}")


In [ ]:
train_ld, val_ld, test_ld = make_loaders(records_slice, augment_train=False)
cw_slice = get_class_weights(records_slice)

slice_metrics = run_experiment(
    model          = ConcatCNNAttention(),
    train_loader   = train_ld,
    val_loader     = val_ld,
    test_loader    = test_ld,
    save_dir       = RESULTS_DIR / "slice_split",
    model_name     = "PaperModel_SliceSplit",
    class_weights  = cw_slice,
)

print(f"\n🚨 Slice Split — Test Accuracy : {slice_metrics['accuracy']*100:.2f}%")
print(f"   F1-Macro                    : {slice_metrics['f1_macro']:.4f}")
print(f"   Benign Recall               : {slice_metrics['per_class']['Benign']['recall']:.4f}")


---
## 5 · Experiment 1B — Sequential Split *(Honest Baseline)* ✅

Loads `splits.csv` from `00_dataset/preprocessing.ipynb` (sequential 70/15/15 per class).
All slices from adjacent CT volumes stay in the same partition → **no leakage**.


In [ ]:
print("=" * 68)
print("✅ EXPERIMENT 1B — Sequential Per-Class Split (Honest Evaluation)")
print("   Expected: ~80–87% accuracy — this is the REAL result")
print("=" * 68)

if SPLITS_CSV.exists():
    print(f"  Loading splits.csv from {SPLITS_CSV.resolve()}")
    records_seq = load_splits_csv()
else:
    print("  splits.csv not found — run 00_dataset/preprocessing.ipynb first!")
    records_all = build_case_registry()
    records_seq = sequential_split(records_all)

print("\nSplit distribution:")
for split in ['train', 'val', 'test']:
    sub = [r for r in records_seq if r['split'] == split]
    print(f"  {split:<6}: {len(sub)} imgs | {dict(Counter(r['class_name'] for r in sub))}")


In [ ]:
train_ld, val_ld, test_ld = make_loaders(records_seq, augment_train=False)
cw_seq = get_class_weights(records_seq)

patient_metrics = run_experiment(
    model          = ConcatCNNAttention(),
    train_loader   = train_ld,
    val_loader     = val_ld,
    test_loader    = test_ld,
    save_dir       = RESULTS_DIR / "sequential_split",
    model_name     = "PaperModel_SequentialSplit",
    class_weights  = cw_seq,
)

print(f"\n✅ Sequential Split — Test Accuracy : {patient_metrics['accuracy']*100:.2f}%")
print(f"   F1-Macro                        : {patient_metrics['f1_macro']:.4f}")
print(f"   Benign Recall                   : {patient_metrics['per_class']['Benign']['recall']:.4f}")


---
## 6 · Leakage Comparison Table

> This table becomes **Table 1** in your thesis — the core leakage demonstration.


In [ ]:
print("\n" + "=" * 70)
print("📊 LEAKAGE DEMONSTRATION — KEY COMPARISON")
print("=" * 70)
print(f"{'Metric':<30} {'Slice Split (Leaked)':>22} {'Sequential (Honest)':>22}")
print("-" * 74)

for key, label in [('accuracy','Test Accuracy'), ('f1_macro','F1-Macro')]:
    s = slice_metrics[key]*100
    p = patient_metrics[key]*100
    print(f"  {label:<28} {s:>21.2f}%  {p:>21.2f}%")

br_s = slice_metrics['per_class']['Benign']['recall']*100
br_p = patient_metrics['per_class']['Benign']['recall']*100
print(f"  {'Benign Recall':<28} {br_s:>21.2f}%  {br_p:>21.2f}%")
print("=" * 70)
print(f"  ΔAccuracy  = {slice_metrics['accuracy']*100 - patient_metrics['accuracy']*100:+.2f} pp (leakage inflation)")
print(f"  ΔF1-Macro  = {(slice_metrics['f1_macro'] - patient_metrics['f1_macro'])*100:+.2f} pp")
print("\n→  The gap is the DATA LEAKAGE EFFECT.")
print("   Slice-level split mixes patient slices across train/test → inflated metrics.")
